In [1]:
# !pip install transformers datasets evaluate accelerate torch fugashi ipadic

In [2]:
import json

with open("tanka.json", encoding="utf8") as f:
  data = json.load(f)

for item in data:
  for key in item['emotion']:
    item['emotion'][key] = \
      0 if item['emotion'][key] <= 3 else \
      1 if item['emotion'][key] <= 6 else 2

In [3]:
from datasets import Dataset

to_learn = ["emotion", "tag"]

label_names = []
for i in to_learn:
    for j in data[0][i]:
        label = i + "." + j
        label_names.append(label)
print(f'label order: {label_names}')

label_list_of_name = {x: set() for x in label_names}
for item in data:
  for i in to_learn:
    for j in item[i]:
      label_list_of_name[i + "." + j].add(item[i][j])
is_binary = {}
for name in label_list_of_name:
    is_binary[name] = (len(label_list_of_name[name]) <= 2)

new_data = { "text": [], "label": [] }
for item in data:
  new_data["text"].append(item["content"])
  labels = []
  for i in to_learn:
    for j in item[i]:
      name = i + "." + j
      if is_binary[name]:
        if item[i][j] == 0: continue
      else:
        name += "." + str(item[i][j])
      labels.append(name)
  new_data["label"].append(labels)

label_list = []
for name in label_names:
  if is_binary[name]:
    label_list.append(name)
  else:
    for i in sorted(label_list_of_name[name]):
        label_list.append(name + "." + str(i))

dataset_raw = Dataset.from_dict(new_data)
print(f'label_list: {label_list}')

label order: ['emotion.happy', 'emotion.funny', 'emotion.calm', 'emotion.sad', 'emotion.lonely', 'emotion.angry', 'tag.daily', 'tag.relationship', 'tag.work', 'tag.life', 'tag.family', 'tag.love', 'tag.travel', 'tag.nature', 'tag.current']
label_list: ['emotion.happy.0', 'emotion.happy.1', 'emotion.happy.2', 'emotion.funny.0', 'emotion.funny.1', 'emotion.funny.2', 'emotion.calm.0', 'emotion.calm.1', 'emotion.calm.2', 'emotion.sad.0', 'emotion.sad.1', 'emotion.sad.2', 'emotion.lonely.0', 'emotion.lonely.1', 'emotion.lonely.2', 'emotion.angry.0', 'emotion.angry.1', 'emotion.angry.2', 'tag.daily', 'tag.relationship', 'tag.work', 'tag.life', 'tag.family', 'tag.love', 'tag.travel', 'tag.nature', 'tag.current']


In [4]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="train",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [5]:
import transformers
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EvalPrediction,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)

model_name = "tohoku-nlp/bert-base-japanese-char"

# Load pretrained model and tokenizer
# In distributed training, the .from_pretrained methods guarantee that only one local process can concurrently
# download model & vocab.
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=len(label_list),
    finetuning_task="text-classification",
    problem_type = "multi_label_classification",
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    config=config,
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at tohoku-nlp/bert-base-japanese-char and are newly initialized: ['classifier.bias', 'classifier.weight'

In [6]:
label_to_id = {v: i for i, v in enumerate(label_list)}
model.config.label2id = label_to_id
model.config.id2label = {id: label for label, id in label_to_id.items()}

In [7]:
def multi_labels_to_ids(labels):
    ids = [0.0] * len(label_to_id)  # BCELoss requires float as target type
    for label in labels:
        ids[label_to_id[label]] = 1.0
    return ids

def preprocess_function(examples):
    # Tokenize the texts
    result = tokenizer(examples["text"])
    if label_to_id is not None and "label" in examples:
        result["label"] = [multi_labels_to_ids(l) for l in examples["label"]]
    return result

# Running the preprocessing pipeline on all the datasets
with training_args.main_process_first(desc="dataset map pre-processing"):
    dataset = dataset_raw.map(
        preprocess_function,
        batched=True
    )

Parameter 'function'=<function preprocess_function at 0x791427227ac0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/149 [00:00<?, ? examples/s]

In [8]:
dataset = dataset.train_test_split(test_size=0.2)

In [9]:
import evaluate
import numpy as np

metric = evaluate.load("f1", config_name="multilabel")

def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    preds = np.array([np.where(p > 0, 1, 0) for p in preds])  # convert logits to multi-hot encoding
    # Micro F1 is commonly used in multi-label classification
    result = metric.compute(predictions=preds, references=p.label_ids, average="micro")
    if len(result) > 1:
        result["combined_score"] = np.mean(list(result.values())).item()
    return result

In [10]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [11]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,No log,0.644493,0.354930
2,No log,0.627931,0.334328


In [12]:
metrics = train_result.metrics
max_train_samples = len(dataset["train"])
metrics["train_samples"] = min(max_train_samples, len(dataset["train"]))
trainer.save_model()  # Saves the tokenizer too for easy upload
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

***** train metrics *****
  epoch                    =        2.0
  total_flos               =     3735GF
  train_loss               =     0.6517
  train_runtime            = 0:02:17.57
  train_samples            =        119
  train_samples_per_second =       1.73
  train_steps_per_second   =      0.116


In [1]:
from datasets import load_dataset

predict_dataset = load_dataset("text", data_files="rest.txt")["train"]
predict_dataset = predict_dataset.map(
    preprocess_function,
    batched=True
)

/home/yamashita/disk02/utakai/venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'preprocess_function' is not defined

In [14]:
import os

predictions = trainer.predict(predict_dataset, metric_key_prefix="predict").predictions

In [15]:
import csv

output_predict_file = os.path.join(training_args.output_dir, "predict_results.csv")
if trainer.is_world_process_zero():
    with open(output_predict_file, "w") as f:
        writer = csv.writer(f)
        writer.writerow(["index", "content"] + label_names)
        for i, item in enumerate(predictions):
            # recover from multi-hot encoding
            value = {x: (0, None) for x in label_names}
            for j in range(len(label_list)):
                for name in label_names:
                    if label_list[j].startswith(name):
                        if is_binary[name]:
                            value[name] = (1 if item[j] > 0 else 0, None)
                        elif value[name][1] == None or value[name][1] < item[j]:
                            value[name] = (label_list[j][-1], item[j])
                        break
            res = [i, predict_dataset[i]["text"]] + [value[name][0] for name in label_names]
            writer.writerow(res)